# FlatFair Ensemble — Dual-Model Buy Recommendation

This notebook combines predictions from two independent models to recommend whether it's **worth buying** an HDB resale flat:

1. **LightGBM** (gradient boosting) — baseline model with known best params
2. **Random Forest** (bagging) — independently tuned model

## How It Works

1. User inputs: **town**, **flat_type**, **budget** via widgets above
2. Feature engineering builds the latest feature row for the selected town/flat_type
3. Both models predict the median resale price
4. **Counter-check 1**: Validate inputs (town, flat_type, budget must be valid)
5. **Counter-check 2**: Models must agree within 20% — if not, recommendation is "Inconclusive"
6. Final recommendation: Worth Buying / Over Budget / Wait / Inconclusive

## Recommendation Logic

| Condition | Recommendation |
|---|---|
| Both models agree & predicted price \u2264 budget | \u2705 Worth Buying |
| Both models agree & predicted price > budget | \u26a0\ufe0f Over Budget |
| Both models agree & price momentum strongly negative | \u23f3 Wait — Prices Declining |
| Models disagree > 20% | \u2754 Inconclusive — Manual Review |

In [0]:
# --- User Input Widgets ---
# Key parameters identified by both models as important for prediction

dbutils.widgets.text("budget", "500000", "Your Budget (S$)")
dbutils.widgets.dropdown("town", "TAMPINES", [
    "ANG MO KIO", "BEDOK", "BISHAN", "BUKIT BATOK", "BUKIT MERAH", "BUKIT PANJANG",
    "BUKIT TIMAH", "CENTRAL AREA", "CHOA CHU KANG", "CLEMENTI", "GEYLANG", "HOUGANG",
    "JURONG EAST", "JURONG WEST", "KALLANG/WHAMPOA", "MARINE PARADE", "PASIR RIS",
    "PUNGGOL", "QUEENSTOWN", "SEMBAWANG", "SENGKANG", "SERANGOON", "TAMPINES",
    "TOA PAYOH", "WOODLANDS", "YISHUN"
], "Town")
dbutils.widgets.dropdown("flat_type", "4 ROOM", [
    "1 ROOM", "2 ROOM", "3 ROOM", "4 ROOM", "5 ROOM", "EXECUTIVE", "MULTI-GENERATION"
], "Flat Type")

dbutils.widgets.dropdown("model_agreement_threshold", "20", [
    "10", "15", "20", "25", "30"
], "Model Agreement Threshold (%)")

print("\u2705 Widgets configured. Adjust the parameters above to get your buy recommendation.")

✅ Widgets configured. Adjust the parameters above to get your buy recommendation.


In [0]:
%pip install lightgbm -q
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import pandas as pd
import numpy as np
from lightgbm import LGBMRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# --- Feature Engineering (same pipeline as both model notebooks) ---
SCHEMA = "workspace.flatfair"
df_txn = spark.table(f"{SCHEMA}.hdb_resale_transactions")

df_monthly = df_txn.groupBy("month", "town", "flat_type").agg(
    F.count("*").alias("txn_count"),
    F.expr("percentile(resale_price, 0.5)").alias("median_price"),
    F.round(F.avg("resale_price")).alias("avg_price"),
    F.round(F.avg("price_per_sqm"), 2).alias("avg_price_per_sqm"),
    F.round(F.avg("floor_area_sqm"), 1).alias("avg_floor_area"),
    F.round(F.avg("remaining_lease_years"), 1).alias("avg_remaining_lease")
)

df_monthly = df_monthly.withColumn("year", F.substring("month", 1, 4).cast("int")) \
    .withColumn("month_num", F.substring("month", 6, 2).cast("int")) \
    .withColumn("quarter", F.ceil(F.col("month_num") / 3).cast("int")) \
    .withColumn("month_idx", F.row_number().over(Window.orderBy("month")) - 1)

w = Window.partitionBy("town", "flat_type").orderBy("month")
for lag in [1, 2, 3, 6, 12]:
    df_monthly = df_monthly.withColumn(f"price_lag_{lag}", F.lag("median_price", lag).over(w))
df_monthly = df_monthly.withColumn("txn_lag_1", F.lag("txn_count", 1).over(w))

w_range_3 = w.rowsBetween(-2, 0)
w_range_6 = w.rowsBetween(-5, 0)
df_monthly = df_monthly \
    .withColumn("price_rolling_3m", F.avg("median_price").over(w_range_3)) \
    .withColumn("price_rolling_6m", F.avg("median_price").over(w_range_6)) \
    .withColumn("txn_rolling_3m", F.avg("txn_count").over(w_range_3))

df_monthly = df_monthly.withColumn(
    "price_momentum",
    F.round((F.col("median_price") - F.col("price_lag_1")) / F.col("price_lag_1") * 100, 2)
)

df_income = spark.table(f"{SCHEMA}.income_by_dwelling_type")
income_map = (
    F.when(F.col("flat_type").isin("1 ROOM", "2 ROOM"), F.col("hdb_1_2_room_income"))
    .when(F.col("flat_type") == "3 ROOM", F.col("hdb_3_room_income"))
    .when(F.col("flat_type") == "4 ROOM", F.col("hdb_4_room_income"))
    .when(F.col("flat_type").isin("5 ROOM", "EXECUTIVE", "MULTI-GENERATION"), F.col("hdb_5_room_exec_income"))
    .otherwise(F.lit(None))
)
df_monthly = df_monthly.join(df_income, "year", "left").withColumn("flat_type_income", income_map)

df_pop = spark.table(f"{SCHEMA}.population_by_town")
df_monthly = df_monthly.join(df_pop, "town", "left")

df_supply = spark.table(f"{SCHEMA}.hdb_supply_pipeline")
df_supply = df_supply.withColumn("supply_year", F.regexp_extract(F.col("year"), r"\d{4}", 0).cast("int")) \
    .withColumnRenamed("year", "supply_fy_raw") \
    .withColumnRenamed("bto_units_launched", "bto_launched") \
    .withColumnRenamed("bto_units_completed", "bto_completed") \
    .withColumnRenamed("total_flats_delivered", "flats_delivered")
df_monthly = df_monthly.join(df_supply, df_monthly.year == df_supply.supply_year, "left").drop("supply_year")
df_monthly = df_monthly \
    .withColumn("bto_launched", F.regexp_replace(F.col("bto_launched"), ",", "").cast("int")) \
    .withColumn("bto_completed", F.regexp_replace(F.col("bto_completed"), ",", "").cast("int")) \
    .withColumn("flats_delivered", F.regexp_replace(F.col("flats_delivered"), ",", "").cast("int"))

df_monthly = df_monthly.withColumn(
    "is_covid_period", F.when(F.col("month").between("2020-04", "2021-03"), 1).otherwise(0)
)

feature_cols = [
    "month", "town", "flat_type", "median_price", "txn_count",
    "year", "month_num", "quarter", "month_idx",
    "avg_price_per_sqm", "avg_floor_area", "avg_remaining_lease",
    "price_lag_1", "price_lag_2", "price_lag_3", "price_lag_6", "price_lag_12",
    "txn_lag_1", "price_rolling_3m", "price_rolling_6m", "txn_rolling_3m",
    "price_momentum", "flat_type_income", "population_2023_est",
    "bto_launched", "bto_completed", "flats_delivered", "is_covid_period"
]
df_features = df_monthly.select(*feature_cols).orderBy("month", "town", "flat_type")
df_features = df_features.filter(F.col("price_lag_12").isNotNull() & F.col("price_lag_1").isNotNull())

pdf = df_features.toPandas()
print(f"Total rows: {len(pdf):,}")
print(f"Date range: {pdf['month'].min()} to {pdf['month'].max()}")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Total rows: 11,701
Date range: 2018-01 to 2026-09


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# --- Train both models with known best hyperparameters ---
cat_features = ["town", "flat_type"]
num_features = [
    "year", "month_num", "quarter", "month_idx",
    "price_lag_1", "price_lag_2", "price_lag_3", "price_lag_6", "price_lag_12",
    "price_rolling_3m", "price_rolling_6m", "price_momentum",
    "flat_type_income", "population_2023_est",
    "bto_launched", "bto_completed", "flats_delivered", "is_covid_period"
]
target = "median_price"
all_features = cat_features + num_features

split_month = "2025-09"
train_df = pdf[pdf["month"] <= split_month].copy()
test_df = pdf[pdf["month"] > split_month].copy()
X_train = train_df[all_features].copy()
y_train = train_df[target].copy()
X_test = test_df[all_features].copy()
y_test = test_df[target].copy()

def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            ("cat", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), cat_features),
            ("num", SimpleImputer(strategy="median"), num_features)
        ],
        remainder="drop"
    )

# Model 1: LightGBM (baseline best params)
lgbm_params = {
    "n_estimators": 500, "learning_rate": 0.05, "max_depth": 8, "num_leaves": 63,
    "min_child_samples": 20, "subsample": 0.8, "subsample_freq": 1,
    "colsample_bytree": 0.8, "reg_alpha": 0.1, "reg_lambda": 0.1
}
lgbm_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("regressor", LGBMRegressor(random_state=42, verbose=-1, **lgbm_params))
])
lgbm_pipeline.fit(X_train, y_train)
lgbm_pred = lgbm_pipeline.predict(X_test)
lgbm_mae = mean_absolute_error(y_test, lgbm_pred)
print(f"\u2705 LightGBM trained — Test MAE: {lgbm_mae:,.0f}")

# Model 2: Random Forest (tuned best params)
rf_params = {
    "n_estimators": 500, "max_depth": 20, "min_samples_split": 5,
    "min_samples_leaf": 2, "max_features": 1.0, "max_samples": 0.7
}
rf_pipeline = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("regressor", RandomForestRegressor(random_state=42, n_jobs=-1, **rf_params))
])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_pred)
print(f"\u2705 Random Forest trained — Test MAE: {rf_mae:,.0f}")

# Weights based on inverse MAE (lower MAE = higher weight)
w_lgbm = 1 / lgbm_mae
w_rf = 1 / rf_mae
w_total = w_lgbm + w_rf
weight_lgbm = w_lgbm / w_total
weight_rf = w_rf / w_total
print(f"\nEnsemble weights (inverse-MAE):")
print(f"  LightGBM:      {weight_lgbm:.1%}")
print(f"  Random Forest: {weight_rf:.1%}")

✅ LightGBM trained — Test MAE: 6,677
✅ Random Forest trained — Test MAE: 9,176

Ensemble weights (inverse-MAE):
  LightGBM:      57.9%
  Random Forest: 42.1%


In [0]:
# ============================================================
# COUNTER-CHECK 1: Validate user inputs before running prediction
# ============================================================

selected_town = dbutils.widgets.get("town")
selected_flat_type = dbutils.widgets.get("flat_type")
budget_str = dbutils.widgets.get("budget")
agreement_threshold = float(dbutils.widgets.get("model_agreement_threshold"))

print("=" * 60)
print("COUNTER-CHECK 1: Input Validation")
print("=" * 60)

validation_errors = []

# Check 1a: Town must exist in data
valid_towns = set(pdf["town"].unique())
if selected_town not in valid_towns:
    validation_errors.append(f"\u274c Invalid town: '{selected_town}'. Must be one of: {sorted(valid_towns)}")
else:
    print(f"\u2705 Town: {selected_town}")

# Check 1b: Flat type must exist in data
valid_flat_types = set(pdf["flat_type"].unique())
if selected_flat_type not in valid_flat_types:
    validation_errors.append(f"\u274c Invalid flat type: '{selected_flat_type}'. Must be one of: {sorted(valid_flat_types)}")
else:
    print(f"\u2705 Flat type: {selected_flat_type}")

# Check 1c: Budget must be a positive number
try:
    budget = float(budget_str)
    if budget <= 0:
        validation_errors.append(f"\u274c Budget must be positive, got: {budget}")
    elif budget < 50000:
        validation_errors.append(f"\u26a0\ufe0f Budget S${budget:,.0f} is unusually low for HDB flats (typical range: S$140K–S$1.7M)")
    elif budget > 2000000:
        validation_errors.append(f"\u26a0\ufe0f Budget S${budget:,.0f} is unusually high for HDB flats (typical range: S$140K–S$1.7M)")
    else:
        print(f"\u2705 Budget: S${budget:,.0f}")
except ValueError:
    validation_errors.append(f"\u274c Budget must be a number, got: '{budget_str}'")
    budget = 0

# Check 1d: Town + flat_type combination must have historical data
combo_data = pdf[(pdf["town"] == selected_town) & (pdf["flat_type"] == selected_flat_type)]
if len(combo_data) == 0:
    validation_errors.append(f"\u274c No historical data for {selected_town} {selected_flat_type}. This combination may not exist.")
else:
    print(f"\u2705 Historical data: {len(combo_data)} months available for {selected_town} {selected_flat_type}")

# Check 1e: Agreement threshold must be reasonable
if agreement_threshold < 5 or agreement_threshold > 50:
    validation_errors.append(f"\u274c Agreement threshold must be between 5% and 50%, got: {agreement_threshold}%")
else:
    print(f"\u2705 Model agreement threshold: {agreement_threshold}%")

if validation_errors:
    print("\n" + "=" * 60)
    print("\u274c COUNTER-CHECK 1 FAILED — Cannot proceed with prediction")
    print("=" * 60)
    for err in validation_errors:
        print(f"  {err}")
    raise ValueError(f"Input validation failed with {len(validation_errors)} error(s). Fix the parameters above and re-run.")
else:
    print("\n" + "=" * 60)
    print("\u2705 COUNTER-CHECK 1 PASSED — All inputs valid")
    print("=" * 60)

COUNTER-CHECK 1: Input Validation
✅ Town: TAMPINES
✅ Flat type: 4 ROOM
✅ Budget: S$500,000
✅ Historical data: 105 months available for TAMPINES 4 ROOM
✅ Model agreement threshold: 20.0%

✅ COUNTER-CHECK 1 PASSED — All inputs valid


In [0]:
# --- Get latest features for user's selection and run both models ---

# Get the most recent month's features for this town/flat_type
latest_row = combo_data.sort_values("month").iloc[-1:][all_features].copy()
latest_month = combo_data.sort_values("month").iloc[-1]["month"]
latest_price = combo_data.sort_values("month").iloc[-1]["median_price"]
latest_momentum = combo_data.sort_values("month").iloc[-1]["price_momentum"]

print(f"Latest data point: {selected_town} {selected_flat_type} — {latest_month}")
print(f"  Latest median price: S${latest_price:,.0f}")
print(f"  Price momentum: {latest_momentum:+.2f}%")

# Run both models on the latest features
lgbm_prediction = lgbm_pipeline.predict(latest_row)[0]
rf_prediction = rf_pipeline.predict(latest_row)[0]

# Weighted ensemble prediction
ensemble_prediction = weight_lgbm * lgbm_prediction + weight_rf * rf_prediction

print(f"\n--- Model Predictions ---")
print(f"  LightGBM:      S${lgbm_prediction:,.0f}")
print(f"  Random Forest: S${rf_prediction:,.0f}")
print(f"  Ensemble (weighted): S${ensemble_prediction:,.0f}")

Latest data point: TAMPINES 4 ROOM — 2026-09
  Latest median price: S$658,000
  Price momentum: -0.30%

--- Model Predictions ---
  LightGBM:      S$662,768
  Random Forest: S$661,015
  Ensemble (weighted): S$662,030


In [0]:
# ============================================================
# COUNTER-CHECK 2: Validate model agreement before recommending
# ============================================================

print("=" * 60)
print("COUNTER-CHECK 2: Model Agreement Validation")
print("=" * 60)

# Calculate percentage difference between models
price_diff = abs(lgbm_prediction - rf_prediction)
pct_diff = (price_diff / min(lgbm_prediction, rf_prediction)) * 100

print(f"  LightGBM prediction:      S${lgbm_prediction:,.0f}")
print(f"  Random Forest prediction:  S${rf_prediction:,.0f}")
print(f"  Difference:                S${price_diff:,.0f} ({pct_diff:.1f}%)")
print(f"  Agreement threshold:       {agreement_threshold}%")

models_agree = pct_diff <= agreement_threshold

if models_agree:
    print(f"\n\u2705 COUNTER-CHECK 2 PASSED — Models agree within {agreement_threshold}%")
else:
    print(f"\n\u274c COUNTER-CHECK 2 FAILED — Models disagree by {pct_diff:.1f}% (threshold: {agreement_threshold}%)")
    print("  The two models have significantly different predictions.")
    print("  Recommendation will be marked as INCONCLUSIVE.")

print("=" * 60)

COUNTER-CHECK 2: Model Agreement Validation
  LightGBM prediction:      S$662,768
  Random Forest prediction:  S$661,015
  Difference:                S$1,753 (0.3%)
  Agreement threshold:       20.0%

✅ COUNTER-CHECK 2 PASSED — Models agree within 20.0%


In [0]:
# ============================================================
# GENERATE FINAL RECOMMENDATION
# ============================================================

print("\n" + "=" * 60)
print("  FlatFair Buy Recommendation")
print("=" * 60)
print(f"  Town:       {selected_town}")
print(f"  Flat Type:  {selected_flat_type}")
print(f"  Budget:     S${budget:,.0f}")
print(f"  Data as of: {latest_month}")
print("=" * 60)

print(f"\n--- Price Predictions ---")
print(f"  LightGBM:          S${lgbm_prediction:,.0f}")
print(f"  Random Forest:     S${rf_prediction:,.0f}")
print(f"  Ensemble:          S${ensemble_prediction:,.0f}")
print(f"  Latest Actual:     S${latest_price:,.0f}")
print(f"  Price Momentum:    {latest_momentum:+.2f}%")

# Determine recommendation
if not models_agree:
    recommendation = "\u2754 INCONCLUSIVE"
    reason = f"Models disagree by {pct_diff:.1f}% (threshold: {agreement_threshold}%). Manual review needed."
    action = "Review the market manually or wait for more data."
elif latest_momentum < -5:
    recommendation = "\u23f3 WAIT — Prices Declining"
    reason = f"Price momentum is {latest_momentum:.1f}%, indicating declining prices."
    action = "Consider waiting for prices to stabilise before buying."
elif ensemble_prediction <= budget:
    surplus = budget - ensemble_prediction
    recommendation = "\u2705 WORTH BUYING"
    reason = f"Predicted price (S${ensemble_prediction:,.0f}) is within your budget (S${budget:,.0f})."
    action = f"You have S${surplus:,.0f} surplus. Both models agree on the price range."
else:
    shortfall = ensemble_prediction - budget
    recommendation = "\u26a0\ufe0f OVER BUDGET"
    reason = f"Predicted price (S${ensemble_prediction:,.0f}) exceeds your budget (S${budget:,.0f})."
    action = f"You need S${shortfall:,.0f} more. Consider a cheaper flat type or different town."

# Affordability ratio
affordability_ratio = budget / ensemble_prediction if ensemble_prediction > 0 else 0

print(f"\n--- RECOMMENDATION ---")
print(f"  Status:   {recommendation}")
print(f"  Reason:   {reason}")
print(f"  Action:   {action}")
print(f"  Budget vs Predicted: {affordability_ratio:.2f}x (1.0 = exactly at budget)")
print("=" * 60)

# Summary table
summary_data = {
    "Parameter": ["Town", "Flat Type", "Budget", "LightGBM Prediction", "RF Prediction", "Ensemble Prediction", "Latest Actual Price", "Price Momentum", "Model Agreement", "Recommendation"],
    "Value": [
        selected_town, selected_flat_type, f"S${budget:,.0f}",
        f"S${lgbm_prediction:,.0f}", f"S${rf_prediction:,.0f}", f"S${ensemble_prediction:,.0f}",
        f"S${latest_price:,.0f}", f"{latest_momentum:+.2f}%",
        f"{pct_diff:.1f}% ({'PASS' if models_agree else 'FAIL'})",
        recommendation
    ]
}
display(pd.DataFrame(summary_data))


  FlatFair Buy Recommendation
  Town:       TAMPINES
  Flat Type:  4 ROOM
  Budget:     S$500,000
  Data as of: 2026-09

--- Price Predictions ---
  LightGBM:          S$662,768
  Random Forest:     S$661,015
  Ensemble:          S$662,030
  Latest Actual:     S$658,000
  Price Momentum:    -0.30%

--- RECOMMENDATION ---
  Status:   ⚠️ OVER BUDGET
  Reason:   Predicted price (S$662,030) exceeds your budget (S$500,000).
  Action:   You need S$162,030 more. Consider a cheaper flat type or different town.
  Budget vs Predicted: 0.76x (1.0 = exactly at budget)


Parameter,Value
Town,TAMPINES
Flat Type,4 ROOM
Budget,"S$500,000"
LightGBM Prediction,"S$662,768"
RF Prediction,"S$661,015"
Ensemble Prediction,"S$662,030"
Latest Actual Price,"S$658,000"
Price Momentum,-0.30%
Model Agreement,0.3% (PASS)
Recommendation,⚠️ OVER BUDGET


In [0]:
# --- Generate predictions for ALL town/flat_type combinations and save to UC ---

# Get the latest row for each town x flat_type combination
latest_rows = pdf.sort_values("month").groupby(["town", "flat_type"]).last().reset_index()

print(f"Generating predictions for {len(latest_rows)} town x flat_type combinations...")

X_all = latest_rows[all_features].copy()

# Run both models
lgbm_preds = lgbm_pipeline.predict(X_all)
rf_preds = rf_pipeline.predict(X_all)

# Ensemble predictions
ensemble_preds = weight_lgbm * lgbm_preds + weight_rf * rf_preds

# Build results DataFrame
results_df = pd.DataFrame({
    "town": latest_rows["town"].values,
    "flat_type": latest_rows["flat_type"].values,
    "latest_month": latest_rows["month"].values,
    "latest_actual_price": latest_rows["median_price"].values,
    "price_momentum": latest_rows["price_momentum"].values,
    "lgbm_prediction": lgbm_preds,
    "rf_prediction": rf_preds,
    "ensemble_prediction": ensemble_preds,
    "price_lag_1": latest_rows["price_lag_1"].values,
    "price_rolling_3m": latest_rows["price_rolling_3m"].values,
    "flat_type_income": latest_rows["flat_type_income"].values,
    "population_2023_est": latest_rows["population_2023_est"].values
})

# Calculate model agreement percentage
results_df["model_diff_pct"] = (
    abs(results_df["lgbm_prediction"] - results_df["rf_prediction"]) /
    results_df[["lgbm_prediction", "rf_prediction"]].min(axis=1) * 100
).round(1)

# Flag if models agree within 20%
results_df["models_agree"] = results_df["model_diff_pct"] <= 20

# Round prices
for col in ["latest_actual_price", "lgbm_prediction", "rf_prediction", "ensemble_prediction", "price_lag_1", "price_rolling_3m"]:
    results_df[col] = results_df[col].round(0)

results_df["price_momentum"] = results_df["price_momentum"].round(2)

# Save to Unity Catalog
spark.createDataFrame(results_df).write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.flatfair.buy_recommendations")

print(f"\u2705 Saved {len(results_df)} predictions to workspace.flatfair.buy_recommendations")
print("\nSample predictions:")
display(results_df[["town", "flat_type", "latest_actual_price", "ensemble_prediction", "model_diff_pct", "models_agree"]].head(10))

Generating predictions for 127 town x flat_type combinations...
✅ Saved 127 predictions to workspace.flatfair.buy_recommendations

Sample predictions:


town,flat_type,latest_actual_price,ensemble_prediction,model_diff_pct,models_agree
ANG MO KIO,2 ROOM,331500.0,328847.0,0.4,true
ANG MO KIO,3 ROOM,405000.0,406856.0,0.3,true
ANG MO KIO,4 ROOM,585000.0,593238.0,1.9,true
ANG MO KIO,5 ROOM,925000.0,932865.0,3.0,true
ANG MO KIO,EXECUTIVE,1090000.0,1106666.0,2.6,true
BEDOK,2 ROOM,324444.0,327897.0,0.6,true
BEDOK,3 ROOM,412500.0,412602.0,0.3,true
BEDOK,4 ROOM,545000.0,553614.0,3.3,true
BEDOK,5 ROOM,852444.0,837271.0,0.4,true
BEDOK,EXECUTIVE,1015000.0,1010023.0,1.0,true
